# Commitment Rebuttal Analysis

This notebook reads the rebuttal training outputs and summarizes:

- AUROC and PR-AUC by model, threshold, feature space, and scenario
- target-environment breakdowns
- calibration curves
- false-positive rates at fixed recall levels


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

RESULTS_ROOT = Path('/playpen-ssd/smerrill/deception2/rebuttal/results')
RUN_NAME = 'commitment_threshold_sweep_v1'
RUN_ROOT = RESULTS_ROOT / RUN_NAME
TRAINING_ROOT = RUN_ROOT / 'training'

CONFIG_PATH = TRAINING_ROOT / 'commitment_rebuttal_config.json'
INVENTORY_PATH = TRAINING_ROOT / 'commitment_rebuttal_inventory.csv'
METRICS_PATH = TRAINING_ROOT / 'commitment_rebuttal_metrics.csv'
CALIBRATION_PATH = TRAINING_ROOT / 'commitment_rebuttal_calibration.csv'
FPR_PATH = TRAINING_ROOT / 'commitment_rebuttal_fpr_at_recall.csv'
ERRORS_PATH = TRAINING_ROOT / 'commitment_rebuttal_errors.csv'
PREDICTIONS_PATH = TRAINING_ROOT / 'commitment_rebuttal_predictions.parquet'

assert TRAINING_ROOT.exists(), f'Missing training directory: {TRAINING_ROOT}'
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8')) if CONFIG_PATH.exists() else {}
inventory_df = pd.read_csv(INVENTORY_PATH) if INVENTORY_PATH.exists() else pd.DataFrame()
metrics_df = pd.read_csv(METRICS_PATH) if METRICS_PATH.exists() else pd.DataFrame()
calibration_df = pd.read_csv(CALIBRATION_PATH) if CALIBRATION_PATH.exists() else pd.DataFrame()
fpr_df = pd.read_csv(FPR_PATH) if FPR_PATH.exists() else pd.DataFrame()
errors_df = pd.read_csv(ERRORS_PATH) if ERRORS_PATH.exists() else pd.DataFrame()
predictions_df = pd.read_parquet(PREDICTIONS_PATH) if PREDICTIONS_PATH.exists() else pd.DataFrame()

print('RUN_ROOT:', RUN_ROOT)
print('TRAINING_ROOT:', TRAINING_ROOT)
print('metrics rows:', len(metrics_df))
print('calibration rows:', len(calibration_df))
print('fpr rows:', len(fpr_df))
print('prediction rows:', len(predictions_df))


In [ ]:
config


In [ ]:
inventory_df


## Best Metric Rows

In [ ]:
if metrics_df.empty:
    print('No metric rows found.')
else:
    best_rows = (
        metrics_df.sort_values(['scenario', 'model_bundle_name', 'label_kind', 'tau', 'target_env', 'pr_auc', 'auroc'], ascending=[True, True, True, True, True, False, False])
        .groupby(['scenario', 'model_bundle_name', 'label_kind', 'tau', 'target_env'], as_index=False)
        .head(1)
        .reset_index(drop=True)
    )
    best_rows[['scenario', 'model_bundle_name', 'label_kind', 'tau', 'target_env', 'feature_space', 'eval_kind', 'auroc', 'pr_auc', 'brier', 'row_count', 'example_count']]


## Aggregate Summary

In [ ]:
if metrics_df.empty:
    print('No metric rows found.')
else:
    summary_df = (
        metrics_df.groupby(['scenario', 'model_bundle_name', 'label_kind', 'tau', 'feature_space', 'eval_kind'], as_index=False)
        .agg(
            mean_auroc=('auroc', 'mean'),
            mean_pr_auc=('pr_auc', 'mean'),
            mean_brier=('brier', 'mean'),
            eval_rows=('row_count', 'sum'),
            target_envs=('target_env', 'nunique'),
        )
        .sort_values(['scenario', 'model_bundle_name', 'tau', 'eval_kind', 'mean_pr_auc'], ascending=[True, True, True, True, False])
        .reset_index(drop=True)
    )
    summary_df


## PR-AUC by Target Environment

In [ ]:
if metrics_df.empty:
    print('No metric rows found.')
else:
    plot_df = metrics_df.loc[metrics_df['eval_kind'].astype(str).eq('ood_test')].copy()
    if plot_df.empty:
        print('No OOD rows found.')
    else:
        grouped = (
            plot_df.groupby(['scenario', 'model_bundle_name', 'tau', 'feature_space', 'target_env'], as_index=False)
            .agg(pr_auc=('pr_auc', 'mean'))
        )
        for (scenario, model_bundle_name, tau), subset in grouped.groupby(['scenario', 'model_bundle_name', 'tau']):
            pivot = subset.pivot(index='target_env', columns='feature_space', values='pr_auc')
            plt.figure(figsize=(1.2 * max(4, len(pivot.columns)), 0.8 * max(3, len(pivot.index))))
            sns.heatmap(pivot, annot=True, fmt='.3f', cmap='viridis', vmin=0.0, vmax=1.0)
            plt.title(f'PR-AUC | {scenario} | {model_bundle_name} | tau={tau}')
            plt.xlabel('Feature Space')
            plt.ylabel('Target Environment')
            plt.tight_layout()
            plt.show()


## AUROC by Target Environment

In [ ]:
if metrics_df.empty:
    print('No metric rows found.')
else:
    plot_df = metrics_df.loc[metrics_df['eval_kind'].astype(str).eq('ood_test')].copy()
    if plot_df.empty:
        print('No OOD rows found.')
    else:
        grouped = (
            plot_df.groupby(['scenario', 'model_bundle_name', 'tau', 'feature_space', 'target_env'], as_index=False)
            .agg(auroc=('auroc', 'mean'))
        )
        for (scenario, model_bundle_name, tau), subset in grouped.groupby(['scenario', 'model_bundle_name', 'tau']):
            pivot = subset.pivot(index='target_env', columns='feature_space', values='auroc')
            plt.figure(figsize=(1.2 * max(4, len(pivot.columns)), 0.8 * max(3, len(pivot.index))))
            sns.heatmap(pivot, annot=True, fmt='.3f', cmap='magma', vmin=0.0, vmax=1.0)
            plt.title(f'AUROC | {scenario} | {model_bundle_name} | tau={tau}')
            plt.xlabel('Feature Space')
            plt.ylabel('Target Environment')
            plt.tight_layout()
            plt.show()


## Calibration Curves

In [ ]:
if calibration_df.empty:
    print('No calibration rows found.')
else:
    curve_df = calibration_df.copy()
    curve_df = curve_df.sort_values(['scenario', 'model_bundle_name', 'tau', 'feature_space', 'bin_idx'])
    for (scenario, model_bundle_name, tau), subset in curve_df.groupby(['scenario', 'model_bundle_name', 'tau']):
        plt.figure(figsize=(6, 5))
        plt.plot([0, 1], [0, 1], linestyle='--', color='black', linewidth=1)
        plotted = False
        for feature_space, fs_df in subset.groupby('feature_space'):
            mean_curve = fs_df.groupby('bin_idx', as_index=False).agg(mean_pred=('mean_pred', 'mean'), frac_pos=('frac_pos', 'mean'))
            if mean_curve.empty:
                continue
            plotted = True
            plt.plot(mean_curve['mean_pred'], mean_curve['frac_pos'], marker='o', label=feature_space)
        if plotted:
            plt.title(f'Calibration | {scenario} | {model_bundle_name} | tau={tau}')
            plt.xlabel('Mean predicted probability')
            plt.ylabel('Observed positive rate')
            plt.legend(loc='best')
            plt.tight_layout()
            plt.show()
        else:
            plt.close()


## False-Positive Rate at Fixed Recall

In [ ]:
if fpr_df.empty:
    print('No FPR-at-recall rows found.')
else:
    summary = (
        fpr_df.groupby(['scenario', 'model_bundle_name', 'tau', 'feature_space', 'recall_target'], as_index=False)
        .agg(mean_fpr=('fpr', 'mean'))
        .sort_values(['scenario', 'model_bundle_name', 'tau', 'recall_target', 'mean_fpr'])
    )
    summary


In [ ]:
if fpr_df.empty:
    print('No FPR-at-recall rows found.')
else:
    plot_df = fpr_df.groupby(['scenario', 'model_bundle_name', 'tau', 'feature_space', 'recall_target'], as_index=False).agg(mean_fpr=('fpr', 'mean'))
    for (scenario, model_bundle_name, tau), subset in plot_df.groupby(['scenario', 'model_bundle_name', 'tau']):
        plt.figure(figsize=(8, 4.5))
        sns.barplot(data=subset, x='recall_target', y='mean_fpr', hue='feature_space')
        plt.title(f'FPR at fixed recall | {scenario} | {model_bundle_name} | tau={tau}')
        plt.xlabel('Recall target')
        plt.ylabel('Mean FPR')
        plt.tight_layout()
        plt.show()


## Errors

In [ ]:
errors_df
